# 📊 CodeAlpha Data Science Internship - Task 2
## Unemployment Analysis with Python (COVID-19 Economic Impact Study)

**Author:** CodeAlpha Intern  
**Domain:** Data Science & Exploratory Data Analysis (EDA)  
**Task Objectives:**  
1. Analyze unemployment rate data representing unemployed people percentage across Indian States and Union Territories.
2. Clean, transform, and visualize historical unemployment trends spanning 2019 to 2020.
3. Quantify and investigate the impact of the **COVID-19 pandemic and national lockdown** on employment.
4. Uncover differences in employment shock between **Rural and Urban** economies.
5. Identify regional patterns and formulate evidence-based policy insights.

---


In [ ]:
# Import required libraries
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid", palette="muted")
%matplotlib inline


### 1. Data Cleaning & Integration

In [ ]:
# Load and clean datasets
csv1_path = os.path.join("dataset", "Unemployment in India.csv")
csv2_path = os.path.join("dataset", "Unemployment_Rate_upto_11_2020.csv")

df1 = pd.read_csv(csv1_path)
df1.columns = [c.strip() for c in df1.columns]
df1 = df1.dropna(subset=['Region', 'Date']).copy()
for col in df1.select_dtypes(include='object').columns:
    df1[col] = df1[col].str.strip()
df1['Date'] = pd.to_datetime(df1['Date'], format='%d-%m-%Y')
df1 = df1.sort_values('Date').reset_index(drop=True)

df2 = pd.read_csv(csv2_path)
df2.columns = [c.strip() for c in df2.columns]
if 'Region.1' in df2.columns:
    df2 = df2.rename(columns={'Region.1': 'Macro_Region'})
df2 = df2.dropna(subset=['Region', 'Date']).copy()
for col in df2.select_dtypes(include='object').columns:
    df2[col] = df2[col].str.strip()
df2['Date'] = pd.to_datetime(df2['Date'], format='%d-%m-%Y')
df2 = df2.sort_values('Date').reset_index(drop=True)

print("Dataset 1 Shape:", df1.shape)
print("Dataset 2 Shape:", df2.shape)
df1.head()


### 2. National Unemployment Timeline & COVID Shock Peak

In [ ]:
# National monthly trend
monthly = df1.groupby('Date')['Estimated Unemployment Rate (%)'].mean().reset_index()

plt.figure(figsize=(12, 6))
plt.plot(monthly['Date'], monthly['Estimated Unemployment Rate (%)'], marker='o', color='#d62728', linewidth=2.5)

peak = monthly.loc[monthly['Estimated Unemployment Rate (%)'].idxmax()]
plt.annotate(
    f"Lockdown Peak: {peak['Estimated Unemployment Rate (%)']:.1f}% ({peak['Date'].strftime('%b %Y')})",
    xy=(peak['Date'], peak['Estimated Unemployment Rate (%)']),
    xytext=(peak['Date'] - pd.Timedelta(days=70), peak['Estimated Unemployment Rate (%)'] + 3),
    arrowprops=dict(facecolor='black', shrink=0.08, width=1.5),
    fontweight='bold', bbox=dict(boxstyle="round,pad=0.3", fc="yellow", alpha=0.5)
)

plt.title("National Monthly Unemployment Rate Trend (2019-2020)", fontsize=14, fontweight='bold')
plt.xlabel("Date")
plt.ylabel("Unemployment Rate (%)")
plt.show()


### 3. Rural vs. Urban Employment Disparity

In [ ]:
# Rural vs Urban comparison
area_trend = df1.groupby(['Date', 'Area'])['Estimated Unemployment Rate (%)'].mean().reset_index()

plt.figure(figsize=(12, 6))
sns.lineplot(data=area_trend, x='Date', y='Estimated Unemployment Rate (%)', hue='Area',
             style='Area', markers=True, dashes=False, linewidth=2.5)
plt.title("Rural vs. Urban Unemployment Trajectories", fontsize=14, fontweight='bold')
plt.xlabel("Date")
plt.ylabel("Unemployment Rate (%)")
plt.show()


### 4. State-Wise Analysis (Top 10 vs Bottom 10)

In [ ]:
# Average unemployment rate by State
state_avg = df2.groupby('Region')['Estimated Unemployment Rate (%)'].mean().sort_values(ascending=False)

fig, axes = plt.subplots(1, 2, figsize=(15, 6))
top10 = state_avg.head(10)
sns.barplot(x=top10.values, y=top10.index, ax=axes[0], hue=top10.index, palette='Reds_r', legend=False)
axes[0].set_title("Top 10 States with Highest Avg Unemployment", fontweight='bold')
axes[0].set_xlabel("Unemployment Rate (%)")

bottom10 = state_avg.tail(10).sort_values(ascending=True)
sns.barplot(x=bottom10.values, y=bottom10.index, ax=axes[1], hue=bottom10.index, palette='Greens_r', legend=False)
axes[1].set_title("Top 10 States with Lowest Avg Unemployment", fontweight='bold')
axes[1].set_xlabel("Unemployment Rate (%)")

plt.tight_layout()
plt.show()


### 5. Heatmap: Month-by-Month State Impact

In [ ]:
# Pivot table for state-level heatmap
pivot = df2.pivot_table(
    index='Region',
    columns=df2['Date'].dt.strftime('%b %Y'),
    values='Estimated Unemployment Rate (%)',
    aggfunc='mean'
)
sorted_cols = sorted(pivot.columns, key=lambda x: pd.to_datetime(x, format='%b %Y'))
pivot = pivot[sorted_cols]

plt.figure(figsize=(13, 9))
sns.heatmap(pivot, cmap='YlOrRd', annot=True, fmt=".1f", linewidths=0.5, cbar_kws={'label': 'Unemployment Rate (%)'})
plt.title("State-wise Unemployment Heatmap (Month-by-Month 2020)", fontsize=14, fontweight='bold')
plt.xlabel("Month")
plt.ylabel("State")
plt.show()


### 6. Correlation Analysis & Policy Takeaways

In [ ]:
# Correlation Matrix
numeric_cols = ['Estimated Unemployment Rate (%)', 'Estimated Employed', 'Estimated Labour Participation Rate (%)']
plt.figure(figsize=(7, 5))
sns.heatmap(df2[numeric_cols].corr(), annot=True, cmap='coolwarm', fmt=".3f", linewidths=1.0)
plt.title("Correlation Matrix", fontweight='bold')
plt.show()


### 7. Executive Summary & Policy Recommendations
1. **The Lockdown Surge**: National unemployment peaked drastically during the nationwide lockdown in April-May 2020 (exceeding 23-25%).
2. **Urban Fragility**: Urban areas recorded sharper spikes due to strict closures in service, manufacturing, and retail sectors.
3. **Rural Buffer**: Agriculture and government rural employment guarantee (MGNREGA) acted as critical cushions.
4. **Policy Recommendations**:
   - Establish an **Urban Employment Guarantee Scheme** modeled on MGNREGA.
   - Enhance liquidity facilities and low-interest credit for small and medium enterprises (MSMEs).
   - Accelerate rural digital financial inclusion and regional safety net mechanisms.
